# Thesis Phase 2: Feature Engineering 🍴
**Objective:** Create Baseline and Hybrid datasets.
**Environment:** Local Repository Execution.

In [ ]:
# 1. Setup & Imports (Auto-Install)
import sys
import subprocess
import os

# Ensure SpaCy Model is present
try:
    import spacy
    # Try loading (lightweight check)
    if not spacy.util.is_package("en_core_web_sm"):
        raise OSError("Model not found")
    print("[✓] SpaCy 'en_core_web_sm' is already installed.")
except (ImportError, OSError):
    print("[!] Downloading SpaCy model 'en_core_web_sm' (approx 12MB)...")
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    print("[✓] Download complete.")

import pandas as pd
import spacy
from collections import Counter
from tqdm.notebook import tqdm

# --- CONFIGURATION ---
BASE_DIR = r"E:\Github\repo-phaze7r\geospatial-tagging-thesis"
INPUT_FILE = os.path.join(BASE_DIR, 'datareported', 'final_enriched_dataset.csv')
OUTPUT_DIR = os.path.join(BASE_DIR, 'datareported')

print(f"[*] Base Directory: {BASE_DIR}")
print(f"[*] Input File: {INPUT_FILE}")

In [ ]:
# 2. Load Data
try:
    df = pd.read_csv(INPUT_FILE).fillna("")
    print(f"[*] Loaded {len(df)} records.")
except FileNotFoundError:
    print(f"[!] Input file not found at {INPUT_FILE}")
    print("    Did you run Notebook 1?")

print("[*] Loading SpaCy...")
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
nlp.enable_pipe("tagger")
print("[✓] SpaCy Loaded Successfully.")

In [ ]:
baseline_docs = []
hybrid_candidates_list = []
texts = df['enriched_description'].tolist()

print("[*] Processing Text...")
for doc in tqdm(nlp.pipe(texts, batch_size=200, n_process=1), total=len(texts)):
    # Baseline
    nouns = [token.text.lower() for token in doc if token.pos_ == 'NOUN' and len(token.text) > 2]
    baseline_docs.append(" ".join(nouns))
    # Hybrid
    compounds = []
    for i in range(len(doc) - 1):
        if doc[i].pos_ == 'ADJ' and doc[i+1].pos_ == 'NOUN':
            compounds.append(f"{doc[i].text.lower()}_{doc[i+1].text.lower()}")
    hybrid_candidates_list.append(compounds)

In [ ]:
# Save Baseline
df['text_baseline'] = baseline_docs
df[['osm_id', 'city', 'text_baseline']].to_csv(os.path.join(OUTPUT_DIR, 'dataset_baseline.csv'), index=False)
print("[+] Saved dataset_baseline.csv")

# Save Hybrid (Filter Support > 0.05 or Top K)
all_compounds = [c for sublist in hybrid_candidates_list for c in sublist]
counts = Counter(all_compounds)
top_counts = counts.most_common(200)
allowed = set([c for c, count in top_counts])

final_hyb = []
for clist in hybrid_candidates_list:
    valid = [c for c in clist if c in allowed]
    final_hyb.append(" ".join(valid))

df['text_hybrid'] = final_hyb
df[['osm_id', 'city', 'text_hybrid']].to_csv(os.path.join(OUTPUT_DIR, 'dataset_hybrid.csv'), index=False)
print("[+] Saved dataset_hybrid.csv")